# RQ2 immutable-window rerun

This notebook reruns all 100 Revision 5 questions and all 36 Revision 4 questions for both answer paths. Both paths receive the same ordered, text-hashed evidence window. Gold labels are unavailable to the model and are used only by the verifier.

Before selecting **Runtime → Run all**:

1. Select a T4 GPU or better under **Runtime → Change runtime type**.
2. Add a Colab secret named `GITHUB_TOKEN` with read access to `Wenjia1215/ComplianceGPT_v2`. If the secret is absent, the notebook asks for the token without displaying it.
3. Approve the Google Drive mount. Results are checkpointed after every question, so the same notebook and output folder can resume after a disconnect.


In [1]:
import torch

assert torch.cuda.is_available(), (
    'CUDA GPU not found. Choose Runtime > Change runtime type > T4 GPU, then rerun.'
)
print('GPU:', torch.cuda.get_device_name(0))


GPU: NVIDIA A100-SXM4-40GB


In [3]:
import base64
import getpass
import subprocess
from pathlib import Path

from google.colab import userdata

REPOSITORY = 'Wenjia1215/ComplianceGPT_v2'
BRANCH = 'codex/rq2-immutable-window-v2'
REPO_DIR = Path('/content/ComplianceGPT_v2_rq2')

try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None
if not github_token:
    github_token = getpass.getpass('GitHub token (read-only repository access is sufficient): ')
if not github_token:
    raise RuntimeError('A GitHub token is required to clone the private repository.')

basic = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
auth_option = f'http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}'
remote_url = f'https://github.com/{REPOSITORY}.git'

if not (REPO_DIR / '.git').is_dir():
    result = subprocess.run(
        ['git', '-c', auth_option, 'clone', '--branch', BRANCH, '--single-branch', remote_url, str(REPO_DIR)],
        text=True, capture_output=True
    )
    if result.returncode != 0:
        raise RuntimeError('Repository clone failed: ' + result.stderr)
else:
    local_head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
    remote_line = subprocess.check_output(
        ['git', '-c', auth_option, 'ls-remote', remote_url, f'refs/heads/{BRANCH}'], text=True
    ).strip()
    remote_head = remote_line.split()[0] if remote_line else ''
    if not remote_head or local_head != remote_head:
        raise RuntimeError('The existing clone is not the current runner commit. Start a fresh Colab runtime.')

github_token = None
basic = None
auth_option = None
print('Runner commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip())


GitHub token (read-only repository access is sufficient): ··········


RuntimeError: Repository clone failed: Cloning into '/content/ComplianceGPT_v2_rq2'...
remote: Write access to repository not granted.
fatal: unable to access 'https://github.com/Wenjia1215/ComplianceGPT_v2.git/': The requested URL returned error: 403


In [ ]:
import subprocess
import sys

requirements = REPO_DIR / 'experiments/answerer_comparison/rq2_matched_v2/requirements-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('Dependencies installed.')


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
OUTPUT_DIR = Path('/content/drive/MyDrive/compliancegpt_rq2_matched_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint directory:', OUTPUT_DIR)


In [ ]:
import subprocess
import sys

runner = REPO_DIR / 'experiments/answerer_comparison/rq2_matched_v2/run_matched_rq2.py'
subprocess.run(
    [sys.executable, str(runner), '--repo-root', str(REPO_DIR), '--output-dir', str(OUTPUT_DIR)],
    check=True,
)


In [ ]:
from google.colab import files

result_zip = OUTPUT_DIR.parent / f'{OUTPUT_DIR.name}.zip'
assert result_zip.is_file(), f'Result archive not found: {result_zip}'
print('Downloading:', result_zip)
files.download(str(result_zip))
